# 3. Transaction Classification

## Objective

The objective of this notebook is to develop machine learning
classification models to categorize customer transactions
based on their spending amounts and transaction-related features.

The analysis will include:
- Data inspection and preparation
- Feature engineering
- Target variable creation
- Model training and testing
- Model performance evaluation
- Comparison and interpretation of results

The goal is to understand whether transaction characteristics
can help predict the spending category of a transaction.

In [4]:

import pandas as pd
from pathlib import Path

# Load the cleaned dataset created in Notebook 1
file_path = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "cleaned_transactions.csv"
)

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 records:")
display(df.head())


Dataset loaded successfully!
Dataset shape: (12043, 24)

Column names:
['status', 'card_present_flag', 'bpay_biller_code', 'account', 'currency', 'long_lat', 'txn_description', 'merchant_id', 'merchant_code', 'first_name', 'balance', 'date_original', 'gender', 'age', 'merchant_suburb', 'merchant_state', 'extraction', 'amount', 'transaction_id', 'country', 'customer_id', 'merchant_long_lat', 'movement', 'date']

Data types:
status                   str
card_present_flag    float64
bpay_biller_code         str
account                  str
currency                 str
long_lat                 str
txn_description          str
merchant_id              str
merchant_code        float64
first_name               str
balance              float64
date_original            str
gender                   str
age                    int64
merchant_suburb          str
merchant_state           str
extraction               str
amount               float64
transaction_id           str
country               

,status,card_present_flag,bpay_biller_code,account,currency,long_lat,txn_description,merchant_id,merchant_code,first_name,...,merchant_suburb,merchant_state,extraction,amount,transaction_id,country,customer_id,merchant_long_lat,movement,date
0,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,NaN,Diana,...,Ashmore,QLD,2018-08-01 01:01:15+00:00,16.25,a623070bfead4541a6b0fff8a09e706c,Australia,CUS-2487424745,153.38 -27.99,debit,2018-08-01
1,authorized,0.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,NaN,Diana,...,Sydney,NSW,2018-08-01 01:13:45+00:00,14.19,13270a2a902145da9db4c951e04b51b9,Australia,CUS-2487424745,151.21 -33.87,debit,2018-08-01
2,authorized,1.0,NaN,ACC-1222300524,AUD,151.23 -33.94,POS,835c231d-8cdf-4e96-859d-e9d571760cf0,NaN,Michael,...,Sydney,NSW,2018-08-01 01:26:15+00:00,6.42,feb79e7ecd7048a5a36ec889d1a94270,Australia,CUS-2142601169,151.21 -33.87,debit,2018-08-01
3,authorized,1.0,NaN,ACC-1037050564,AUD,153.10 -27.66,SALES-POS,48514682-c78a-4a88-b0da-2d6302e64673,NaN,Rhonda,...,Buderim,QLD,2018-08-01 01:38:45+00:00,40.90,2698170da3704fd981b15e64a006079e,Australia,CUS-1614226872,153.05 -26.68,debit,2018-08-01
4,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,b4e02c10-0852-4273-b8fd-7b3395e32eb0,NaN,Diana,...,Mermaid Beach,QLD,2018-08-01 01:51:15+00:00,3.25,329adf79878c4cf0aeb4188b4691c266,Australia,CUS-2487424745,153.44 -28.06,debit,2018-08-01


In [5]:
# cell 2 

# 1. Dataset information
print("DATASET INFORMATION")
df.info()

# 2. Missing values
print("\nMISSING VALUES")
missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": (missing_values / len(df)) * 100
})

display(
    missing_summary[missing_summary["Missing Count"] > 0]
    .sort_values(by="Missing Percentage", ascending=False)
)

# 3. Duplicate records
print("\nDUPLICATE RECORDS")
print("Total duplicate rows:", df.duplicated().sum())

# 4. Unique values in categorical columns
print("\nUNIQUE VALUES IN CATEGORICAL COLUMNS")

categorical_columns = df.select_dtypes(
    include=["object", "str", "category"]
).columns

for column in categorical_columns:
    print(f"{column}: {df[column].nunique()} unique values")

# 5. Important categorical distributions
print("\nMOVEMENT DISTRIBUTION")
display(df["movement"].value_counts())

print("\nSTATUS DISTRIBUTION")
display(df["status"].value_counts())

print("\nTRANSACTION DESCRIPTION DISTRIBUTION")
display(df["txn_description"].value_counts().head(15))

# 6. Numerical summary
print("\nNUMERICAL SUMMARY")
display(df[["amount", "balance", "age"]].describe())


DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   status             12043 non-null  str    
 1   card_present_flag  7717 non-null   float64
 2   bpay_biller_code   885 non-null    str    
 3   account            12043 non-null  str    
 4   currency           12043 non-null  str    
 5   long_lat           12043 non-null  str    
 6   txn_description    12043 non-null  str    
 7   merchant_id        7717 non-null   str    
 8   merchant_code      883 non-null    float64
 9   first_name         12043 non-null  str    
 10  balance            12043 non-null  float64
 11  date_original      12043 non-null  str    
 12  gender             12043 non-null  str    
 13  age                12043 non-null  int64  
 14  merchant_suburb    7717 non-null   str    
 15  merchant_state     7717 non-null   str    
 16  extraction   

,Missing Count,Missing Percentage
merchant_code,11160,92.667940
bpay_biller_code,11158,92.651333
card_present_flag,4326,35.921282
merchant_id,4326,35.921282
merchant_suburb,4326,35.921282
merchant_state,4326,35.921282
merchant_long_lat,4326,35.921282



DUPLICATE RECORDS
Total duplicate rows: 0

UNIQUE VALUES IN CATEGORICAL COLUMNS
status: 2 unique values
bpay_biller_code: 3 unique values
account: 100 unique values
currency: 1 unique values
long_lat: 100 unique values
txn_description: 6 unique values
merchant_id: 5725 unique values
first_name: 80 unique values
date_original: 91 unique values
gender: 2 unique values
merchant_suburb: 1609 unique values
merchant_state: 8 unique values
extraction: 9442 unique values
transaction_id: 12043 unique values
country: 1 unique values
customer_id: 100 unique values
merchant_long_lat: 2703 unique values
movement: 2 unique values
date: 91 unique values

MOVEMENT DISTRIBUTION


movement
debit     11160
credit      883
Name: count, dtype: int64


STATUS DISTRIBUTION


status
authorized    7717
posted        4326
Name: count, dtype: int64


TRANSACTION DESCRIPTION DISTRIBUTION


txn_description
SALES-POS     3934
POS           3783
PAYMENT       2600
PAY/SALARY     883
INTER BANK     742
PHONE BANK     101
Name: count, dtype: int64


NUMERICAL SUMMARY


,amount,balance,age
count,12043.000000,12043.000000,12043.000000
mean,187.933588,14704.195553,30.582330
std,592.599934,31503.722652,10.046343
min,0.100000,0.240000,18.000000
25%,16.000000,3158.585000,22.000000
50%,29.000000,6432.010000,28.000000
75%,53.655000,12465.945000,38.000000
max,8835.980000,267128.520000,78.000000


In [7]:
# CELL 3

# 1. Distribution of transaction descriptions
print("TRANSACTION DESCRIPTION DISTRIBUTION")

description_counts = df["txn_description"].value_counts()

description_summary = pd.DataFrame({
    "Count": description_counts,
    "Percentage": (description_counts / len(df)) * 100
})

display(description_summary)

# 2. Relationship between movement and transaction description
print("\nMOVEMENT vs TRANSACTION DESCRIPTION")

display(pd.crosstab(
    df["txn_description"],
    df["movement"],
    margins=True
))

# 3. Relationship between status and transaction description
print("\nSTATUS vs TRANSACTION DESCRIPTION")

display(pd.crosstab(
    df["txn_description"],
    df["status"],
    margins=True
))

# 4. Relationship between movement and status
print("\nMOVEMENT vs STATUS")

display(pd.crosstab(
    df["movement"],
    df["status"],
    margins=True
))

# 5. Amount statistics by transaction description
print("\nAMOUNT STATISTICS BY TRANSACTION DESCRIPTION")

display(
    df.groupby("txn_description")["amount"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_values("count", ascending=False)
)


TRANSACTION DESCRIPTION DISTRIBUTION


,Count,Percentage
txn_description,,
SALES-POS,3934,32.666279
POS,3783,31.412439
PAYMENT,2600,21.589305
PAY/SALARY,883,7.332060
INTER BANK,742,6.161256
PHONE BANK,101,0.838661



MOVEMENT vs TRANSACTION DESCRIPTION


movement,credit,debit,All
txn_description,,,
INTER BANK,0,742,742
PAY/SALARY,883,0,883
PAYMENT,0,2600,2600
PHONE BANK,0,101,101
POS,0,3783,3783
SALES-POS,0,3934,3934
All,883,11160,12043



STATUS vs TRANSACTION DESCRIPTION


status,authorized,posted,All
txn_description,,,
INTER BANK,0,742,742
PAY/SALARY,0,883,883
PAYMENT,0,2600,2600
PHONE BANK,0,101,101
POS,3783,0,3783
SALES-POS,3934,0,3934
All,7717,4326,12043



MOVEMENT vs STATUS


status,authorized,posted,All
movement,,,
credit,0,883,883
debit,7717,3443,11160
All,7717,4326,12043



AMOUNT STATISTICS BY TRANSACTION DESCRIPTION


,count,mean,median,min,max
txn_description,,,,,
SALES-POS,3934,39.909789,20.035,0.1,4233.00
POS,3783,40.407412,19.430,0.1,7081.09
PAYMENT,2600,77.613077,42.500,15.0,1981.00
PAY/SALARY,883,1898.728029,1626.480,576.0,8835.98
INTER BANK,742,86.699461,39.000,16.0,1956.00
PHONE BANK,101,106.099010,43.000,21.0,1916.00


In [8]:
# CELL 4

# Check how card presence relates to transaction description
print("CARD PRESENCE vs TRANSACTION DESCRIPTION")

display(pd.crosstab(
    df["txn_description"],
    df["card_present_flag"],
    margins=True,
    dropna=False
))

# Check whether merchant information is available
# for each transaction description
print("\nMERCHANT ID AVAILABILITY")

df["has_merchant_id"] = df["merchant_id"].notna()

display(pd.crosstab(
    df["txn_description"],
    df["has_merchant_id"],
    normalize="index"
).round(3))

# Check merchant state availability
print("\nMERCHANT STATE AVAILABILITY")

df["has_merchant_state"] = df["merchant_state"].notna()

display(pd.crosstab(
    df["txn_description"],
    df["has_merchant_state"],
    normalize="index"
).round(3))

# Compare amount distributions across descriptions
print("\nAMOUNT DISTRIBUTION BY TRANSACTION DESCRIPTION")

display(
    df.groupby("txn_description")["amount"]
    .quantile([0.25, 0.50, 0.75])
    .unstack()
    .rename(columns={
        0.25: "25th Percentile",
        0.50: "Median",
        0.75: "75th Percentile"
    })
)

# Check date and time fields
print("\nDATE AND TIME SAMPLE")

display(df[["date", "date_original", "extraction"]].head(10))


CARD PRESENCE vs TRANSACTION DESCRIPTION


card_present_flag,0.0,1.0,NaN,All
txn_description,,,,
INTER BANK,0,0,742,742
PAY/SALARY,0,0,883,883
PAYMENT,0,0,2600,2600
PHONE BANK,0,0,101,101
POS,758,3025,0,3783
SALES-POS,765,3169,0,3934
All,1523,6194,4326,12043



MERCHANT ID AVAILABILITY


has_merchant_id,False,True
txn_description,,
INTER BANK,1.0,0.0
PAY/SALARY,1.0,0.0
PAYMENT,1.0,0.0
PHONE BANK,1.0,0.0
POS,0.0,1.0
SALES-POS,0.0,1.0



MERCHANT STATE AVAILABILITY


has_merchant_state,False,True
txn_description,,
INTER BANK,1.0,0.0
PAY/SALARY,1.0,0.0
PAYMENT,1.0,0.0
PHONE BANK,1.0,0.0
POS,0.0,1.0
SALES-POS,0.0,1.0



AMOUNT DISTRIBUTION BY TRANSACTION DESCRIPTION


,25th Percentile,Median,75th Percentile
txn_description,,,
INTER BANK,26.000,39.000,83.000
PAY/SALARY,1013.670,1626.480,2538.680
PAYMENT,32.000,42.500,70.000
PHONE BANK,36.000,43.000,67.000
POS,12.035,19.430,33.155
SALES-POS,12.160,20.035,34.575



DATE AND TIME SAMPLE


,date,date_original,extraction
0,2018-08-01,08-01-2018,2018-08-01 01:01:15+00:00
1,2018-08-01,08-01-2018,2018-08-01 01:13:45+00:00
2,2018-08-01,08-01-2018,2018-08-01 01:26:15+00:00
3,2018-08-01,08-01-2018,2018-08-01 01:38:45+00:00
4,2018-08-01,08-01-2018,2018-08-01 01:51:15+00:00
5,2018-08-01,08-01-2018,2018-08-01 02:00:00+00:00
6,2018-08-01,08-01-2018,2018-08-01 02:23:04+00:00
7,2018-08-01,08-01-2018,2018-08-01 04:11:25+00:00
8,2018-08-01,08-01-2018,2018-08-01 04:40:00+00:00
9,2018-08-01,08-01-2018,2018-08-01 06:00:00+00:00


In [9]:
# CELL 5

# Target distribution
print("MOVEMENT DISTRIBUTION")

display(
    df["movement"].value_counts().to_frame("Count")
)

# Percentage distribution
print("\nMOVEMENT PERCENTAGE")

display(
    (df["movement"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("Percentage")
)

# Amount statistics by movement
print("\nAMOUNT BY MOVEMENT")

display(
    df.groupby("movement")["amount"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

# Balance statistics by movement
print("\nBALANCE BY MOVEMENT")

display(
    df.groupby("movement")["balance"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

# Age statistics by movement
print("\nAGE BY MOVEMENT")

display(
    df.groupby("movement")["age"]
    .agg(["count", "mean", "median", "min", "max"])
)

# Gender distribution by movement
print("\nGENDER vs MOVEMENT")

display(
    pd.crosstab(
        df["gender"],
        df["movement"],
        normalize="columns"
    ).round(3)
)


MOVEMENT DISTRIBUTION


,Count
movement,
debit,11160
credit,883



MOVEMENT PERCENTAGE


,Percentage
movement,
debit,92.67
credit,7.33



AMOUNT BY MOVEMENT


,count,mean,median,std,min,max
movement,,,,,,
credit,883,1898.728029,1626.48,1150.364621,576.0,8835.98
debit,11160,52.572343,26.93,156.354143,0.1,7081.09


... (2 outputs left out)


AGE BY MOVEMENT


,count,mean,median,min,max
movement,,,,,
credit,883,32.631937,30.0,18,78
debit,11160,30.420161,28.0,18,78



GENDER vs MOVEMENT


movement,credit,debit
gender,,
F,0.475,0.478
M,0.525,0.522


In [10]:

# cell 6

# Convert extraction timestamp into datetime
df["extraction_datetime"] = pd.to_datetime(
    df["extraction"],
    utc=True
)

# Sort transactions by customer and timestamp
df_sorted = df.sort_values(
    by=["customer_id", "extraction_datetime"]
).copy()

# Calculate the previous transaction balance for each customer
df_sorted["previous_balance"] = (
    df_sorted.groupby("customer_id")["balance"].shift(1)
)

# Calculate balance change
df_sorted["balance_change"] = (
    df_sorted["balance"] - df_sorted["previous_balance"]
)

# Remove first transaction of each customer
balance_analysis = df_sorted.dropna(
    subset=["previous_balance"]
).copy()

# Compare balance change with transaction amount
print("BALANCE CHANGE BY MOVEMENT")

display(
    balance_analysis.groupby("movement")[
        "balance_change"
    ].agg(["count", "mean", "median", "min", "max"])
)

# Check whether balance changes match transaction direction
balance_analysis["expected_balance_change"] = (
    balance_analysis["amount"].where(
        balance_analysis["movement"] == "credit",
        -balance_analysis["amount"]
    )
)

balance_analysis["balance_difference_error"] = (
    balance_analysis["balance_change"]
    - balance_analysis["expected_balance_change"]
)

print("\nBALANCE CHANGE MATCHING")

display(
    balance_analysis.groupby("movement")[
        "balance_difference_error"
    ].agg(["mean", "median", "std"])
)

print("\nSAMPLE TRANSACTIONS")

display(
    balance_analysis[
        [
            "customer_id",
            "extraction_datetime",
            "movement",
            "amount",
            "previous_balance",
            "balance",
            "balance_change"
        ]
    ].head(15)
)


BALANCE CHANGE BY MOVEMENT


,count,mean,median,min,max
movement,,,,,
credit,872,1892.575390,1626.48,576.00,8835.98
debit,11071,-51.934506,-27.00,-7081.09,3932.27



BALANCE CHANGE MATCHING


,mean,median,std
movement,,,
credit,-3.292764,0.0,61.88610
debit,0.602960,0.0,156.55161



SAMPLE TRANSACTIONS


,customer_id,extraction_datetime,movement,amount,previous_balance,balance,balance_change
272,CUS-1005756958,2018-08-02 21:36:42+00:00,debit,39.95,463.96,424.01,-39.95
315,CUS-1005756958,2018-08-03 08:27:40+00:00,debit,22.96,424.01,401.05,-22.96
320,CUS-1005756958,2018-08-03 09:15:59+00:00,debit,9.11,401.05,391.94,-9.11
435,CUS-1005756958,2018-08-03 21:51:32+00:00,debit,14.65,391.94,377.29,-14.65
456,CUS-1005756958,2018-08-04 04:00:00+00:00,debit,148.00,377.29,229.29,-148.00
464,CUS-1005756958,2018-08-04 05:29:59+00:00,debit,21.77,229.29,207.52,-21.77
548,CUS-1005756958,2018-08-04 21:35:08+00:00,debit,23.61,207.52,183.91,-23.61
572,CUS-1005756958,2018-08-05 03:27:08+00:00,debit,49.13,183.91,134.78,-49.13
598,CUS-1005756958,2018-08-05 10:08:22+00:00,debit,17.89,134.78,116.89,-17.89
690,CUS-1005756958,2018-08-06 09:45:52+00:00,debit,16.66,116.89,100.23,-16.66


In [11]:
# cell 7

# Create a separate modelling dataframe
model_df = df.copy()

# Convert transaction timestamp into datetime
model_df["extraction_datetime"] = pd.to_datetime(
    model_df["extraction"],
    utc=True
)

# Extract useful time-based features
model_df["transaction_hour"] = (
    model_df["extraction_datetime"].dt.hour
)

model_df["day_of_week"] = (
    model_df["extraction_datetime"].dt.dayofweek
)

model_df["transaction_month"] = (
    model_df["extraction_datetime"].dt.month
)

# Select initial predictive features
features = [
    "amount",
    "age",
    "gender",
    "transaction_hour",
    "day_of_week",
    "transaction_month"
]

# Define input and target
X = model_df[features].copy()
y = model_df["movement"].copy()

print("FEATURE DATASET")
display(X.head())

print("\nTARGET DATASET")
display(y.head())

print("\nFEATURE INFORMATION")
print(X.info())

print("\nMISSING VALUES IN FEATURES")
display(X.isnull().sum())

print("\nTARGET DISTRIBUTION")
display(y.value_counts())

print("\nFINAL SHAPES")
print("Features:", X.shape)
print("Target:", y.shape)


FEATURE DATASET


,amount,age,gender,transaction_hour,day_of_week,transaction_month
0,16.25,26,F,1,2,8
1,14.19,26,F,1,2,8
2,6.42,38,M,1,2,8
3,40.90,40,F,1,2,8
4,3.25,26,F,1,2,8



TARGET DATASET


0    debit
1    debit
2    debit
3    debit
4    debit
Name: movement, dtype: str


FEATURE INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   amount             12043 non-null  float64
 1   age                12043 non-null  int64  
 2   gender             12043 non-null  str    
 3   transaction_hour   12043 non-null  int32  
 4   day_of_week        12043 non-null  int32  
 5   transaction_month  12043 non-null  int32  
dtypes: float64(1), int32(3), int64(1), str(1)
memory usage: 435.3 KB
None

MISSING VALUES IN FEATURES


amount               0
age                  0
gender               0
transaction_hour     0
day_of_week          0
transaction_month    0
dtype: int64


TARGET DISTRIBUTION


movement
debit     11160
credit      883
Name: count, dtype: int64


FINAL SHAPES
Features: (12043, 6)
Target: (12043,)


In [12]:
# cell 8

from sklearn.model_selection import StratifiedGroupKFold

# Customer IDs are used only for grouping, not as model features
groups = model_df["customer_id"]

# Create a stratified, customer-aware split
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_index, test_index = next(
    splitter.split(X, y, groups=groups)
)

# Split features and target
X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

# Check customer overlap
train_customers = set(groups.iloc[train_index])
test_customers = set(groups.iloc[test_index])

print("TRAINING DATA SHAPE:", X_train.shape)
print("TESTING DATA SHAPE:", X_test.shape)

print("\nCUSTOMER DISTRIBUTION")
print("Training customers:", len(train_customers))
print("Testing customers:", len(test_customers))
print("Overlapping customers:", len(train_customers & test_customers))

print("\nTRAINING TARGET DISTRIBUTION")
display(y_train.value_counts())
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTESTING TARGET DISTRIBUTION")
display(y_test.value_counts())
display((y_test.value_counts(normalize=True) * 100).round(2))


TRAINING DATA SHAPE: (9598, 6)
TESTING DATA SHAPE: (2445, 6)

CUSTOMER DISTRIBUTION
Training customers: 81
Testing customers: 19
Overlapping customers: 0

TRAINING TARGET DISTRIBUTION


movement
debit     8888
credit     710
Name: count, dtype: int64

movement
debit     92.6
credit     7.4
Name: proportion, dtype: float64


TESTING TARGET DISTRIBUTION


movement
debit     2272
credit     173
Name: count, dtype: int64

movement
debit     92.92
credit     7.08
Name: proportion, dtype: float64

In [13]:
# cell 9
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Separate numerical and categorical features
numerical_features = [
    "amount",
    "age",
    "transaction_hour",
    "day_of_week",
    "transaction_month"
]

categorical_features = ["gender"]

# Numerical preprocessing
numerical_transformer = StandardScaler()

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine both preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

print("\nNumerical features:", numerical_features)
print("Categorical features:", categorical_features)


Preprocessing pipeline created successfully.

Numerical features: ['amount', 'age', 'transaction_hour', 'day_of_week', 'transaction_month']
Categorical features: ['gender']


In [14]:
# cell 10- training the classification model

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# Define the models
models = {
    "Dummy Classifier": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

# Train each model
for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    print(f"{name} trained successfully.")

print("\nAll models have been trained.")



Training Dummy Classifier...
Dummy Classifier trained successfully.

Training Logistic Regression...
Logistic Regression trained successfully.

Training Random Forest...
Random Forest trained successfully.

All models have been trained.


In [15]:
# cell 11 - Evaluate model performance

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report
)

results = []

for name, model in models.items():

    # Predict movement for unseen test transactions
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Credit Precision": precision_score(
            y_test, y_pred, pos_label="credit", zero_division=0
        ),
        "Credit Recall": recall_score(
            y_test, y_pred, pos_label="credit", zero_division=0
        ),
        "Credit F1": f1_score(
            y_test, y_pred, pos_label="credit", zero_division=0
        ),
        "Macro F1": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    })

# Display comparison
results_df = pd.DataFrame(results)

results_df.round(4)


,Model,Accuracy,Balanced Accuracy,Credit Precision,Credit Recall,Credit F1,Macro F1
0,Dummy Classifier,0.9292,0.5000,0.0000,0.0000,0.0000,0.4817
1,Logistic Regression,0.9885,0.9938,0.8607,1.0000,0.9251,0.9595
2,Random Forest,0.9918,0.9636,0.9527,0.9306,0.9415,0.9686


In [16]:
# cell 12- confusion matrices

from sklearn.metrics import confusion_matrix

for name, model in models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=["credit", "debit"]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual Credit", "Actual Debit"],
        columns=["Predicted Credit", "Predicted Debit"]
    )

    print(f"\n{name}")
    display(cm_df)



Dummy Classifier


,Predicted Credit,Predicted Debit
Actual Credit,0,173
Actual Debit,0,2272



Logistic Regression


,Predicted Credit,Predicted Debit
Actual Credit,173,0
Actual Debit,28,2244



Random Forest


,Predicted Credit,Predicted Debit
Actual Credit,161,12
Actual Debit,8,2264


In [17]:
# cell 13 - Investigate influence of transaction amount

from sklearn.base import clone

# Three feature sets
feature_sets = {
    "All Features": features,
    "Amount Only": ["amount"],
    "Without Amount": [
        "age",
        "gender",
        "transaction_hour",
        "day_of_week",
        "transaction_month"
    ]
}

ablation_results = []

for feature_name, selected_features in feature_sets.items():

    X_train_subset = X_train[selected_features]
    X_test_subset = X_test[selected_features]

    # Build a fresh preprocessing pipeline for this feature set
    numerical = [
        col for col in selected_features
        if col != "gender"
    ]

    categorical = [
        col for col in selected_features
        if col == "gender"
    ]

    transformers = []

    if numerical:
        transformers.append(
            ("num", StandardScaler(), numerical)
        )

    if categorical:
        transformers.append(
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
        )

    subset_preprocessor = ColumnTransformer(
        transformers=transformers
    )

    for model_name, classifier in [
        ("Logistic Regression", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )),
        ("Random Forest", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ]:

        pipeline = Pipeline(steps=[
            ("preprocessor", clone(subset_preprocessor)),
            ("classifier", clone(classifier))
        ])

        pipeline.fit(X_train_subset, y_train)

        predictions = pipeline.predict(X_test_subset)

        ablation_results.append({
            "Feature Set": feature_name,
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Credit Precision": precision_score(
                y_test, predictions,
                pos_label="credit",
                zero_division=0
            ),
            "Credit Recall": recall_score(
                y_test, predictions,
                pos_label="credit",
                zero_division=0
            ),
            "Credit F1": f1_score(
                y_test, predictions,
                pos_label="credit",
                zero_division=0
            )
        })

ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df
    .sort_values(["Model", "Feature Set"])
    .round(4)
)


,Feature Set,Model,Accuracy,Credit Precision,Credit Recall,Credit F1
0,All Features,Logistic Regression,0.9885,0.8607,1.0000,0.9251
2,Amount Only,Logistic Regression,0.9890,0.8650,1.0000,0.9276
4,Without Amount,Logistic Regression,0.6004,0.1285,0.8035,0.2215
1,All Features,Random Forest,0.9918,0.9527,0.9306,0.9415
3,Amount Only,Random Forest,0.9881,0.9091,0.9249,0.9169
5,Without Amount,Random Forest,0.8822,0.2051,0.2312,0.2174


In [18]:
# cell 14- feature importance

from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

# Use credit F1 as the scoring metric
credit_f1_scorer = make_scorer(
    f1_score,
    pos_label="credit"
)

importance = permutation_importance(
    models["Random Forest"],
    X_test,
    y_test,
    scoring=credit_f1_scorer,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance.importances_mean,
    "Std": importance.importances_std
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

display(importance_df.round(4))


,Feature,Importance,Std
0,amount,0.8346,0.0169
3,transaction_hour,0.2538,0.0228
4,day_of_week,0.1749,0.0244
5,transaction_month,0.0025,0.0015
1,age,0.0007,0.0026
2,gender,-0.0230,0.0048


In [20]:
# cell 15- model stability using cross validation

from sklearn.model_selection import cross_validate, StratifiedGroupKFold
from sklearn.metrics import make_scorer

# Five-fold customer-aware cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Keep customer groups corresponding to the training rows
train_groups = groups.iloc[train_index]

# Define evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "credit_precision": make_scorer(
        precision_score,
        pos_label="credit",
        zero_division=0
    ),
    "credit_recall": make_scorer(
        recall_score,
        pos_label="credit",
        zero_division=0
    ),
    "credit_f1": make_scorer(
        f1_score,
        pos_label="credit",
        zero_division=0
    )
}

cv_results = []

for name in ["Logistic Regression", "Random Forest"]:

    scores = cross_validate(
        models[name],
        X_train,
        y_train,
        groups=train_groups,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    cv_results.append({
        "Model": name,
        "Mean Accuracy": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "Mean Credit Precision": scores["test_credit_precision"].mean(),
        "Mean Credit Recall": scores["test_credit_recall"].mean(),
        "Mean Credit F1": scores["test_credit_f1"].mean(),
        "Credit F1 Std": scores["test_credit_f1"].std()
    })

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(4))


,Model,Mean Accuracy,Accuracy Std,Mean Credit Precision,Mean Credit Recall,Mean Credit F1,Credit F1 Std
0,Logistic Regression,0.9910,0.0020,0.9059,0.9823,0.9419,0.0124
1,Random Forest,0.9961,0.0028,0.9669,0.9823,0.9741,0.0182


## Conclusion — Transaction Classification

### Objective
The objective of this analysis was to develop machine learning models to classify financial transactions into two categories: debit and credit.

### Data Preparation
- Used the cleaned transaction dataset containing 12,043 records.
- Selected `movement` as the target variable.
- Excluded balance-related features because balance changes can directly reveal transaction movement.
- Created transaction hour, day of week and transaction month features.
- Used a customer-aware train-test split to prevent the same customers from appearing in both sets.

### Models Developed
Three classification approaches were evaluated:
1. Dummy Classifier
2. Logistic Regression
3. Random Forest Classifier

### Model Evaluation
Random Forest achieved the strongest overall results.

- Test Accuracy: 99.18%
- Test Credit Precision: 95.27%
- Test Credit Recall: 93.06%
- Test Credit F1-score: 94.15%
- Cross-validation Mean Credit F1-score: 97.41%

Logistic Regression achieved 100% credit recall on the test set, making it a useful alternative when identifying as many credit transactions as possible is the priority.

### Feature Analysis
Feature ablation and permutation importance showed that transaction amount was the most influential predictor. Transaction hour and day of week also contributed to Random Forest's predictions.

Age, gender and transaction month contributed very little.

### Limitations
- The dataset contains transactions from only 100 customers.
- The data covers a limited historical period.
- Transaction amount dominates the classification process.
- The evaluation tests generalisation to unseen customers, not prediction on future dates.
- Performance should be validated on a larger and more recent dataset before real-world deployment.

### Final Outcome
Random Forest was selected as the preferred classification model based on its overall performance across the test set and cross-validation. Logistic Regression was retained as a comparison model.

The analysis demonstrates how machine learning can identify transaction movement patterns while highlighting the importance of class imbalance, feature selection, data leakage prevention and customer-aware evaluation.